In [3]:
import math
import os
import csv
import json
import torch

from transformer import Tokenizer
from transformer import Transformer

Loading transformer_model.py


In [ ]:
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"

def decode(model, src_sentence, max_len, de_tokenizer, dgs_token2idx, dgs_idx2token, device):
    model.eval()
    # Tokenize source sentence
    src_tokens = de_tokenizer.encode(src_sentence)
    src_tensor = torch.tensor(src_tokens, dtype=torch.long).unsqueeze(0).to(device)  # (1, src_seq_len)
    src_tensor = src_tensor.transpose(0,1)  # (src_seq_len, 1)

    memory = model.encoder(model.pos_encoder(model.src_embedding(src_tensor) * math.sqrt(model.model_dim)))

    # Initialize target sequence with <SOS>
    tgt_indices = [dgs_token2idx[SOS_TOKEN]]
    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt_indices, dtype=torch.long).unsqueeze(1).to(device)  # (tgt_seq_len, 1)
        tgt_tensor = model.pos_decoder(model.tgt_embedding(tgt_tensor) * math.sqrt(model.model_dim))
        tgt_mask = torch.triu(torch.ones((tgt_tensor.size(0), tgt_tensor.size(0)), device=device) == 1).transpose(0, 1)
        tgt_mask = tgt_mask.float().masked_fill(tgt_mask == 0, float('-inf')).masked_fill(tgt_mask == 1, float(0.0))

        out = model.decoder(tgt_tensor, memory, tgt_mask=tgt_mask)
        out = model.fc_out(out)
        # Get last token's prediction
        prob = out[-1, 0]
        next_token = torch.argmax(prob).item()
        tgt_indices.append(next_token)
        if next_token == dgs_token2idx[EOS_TOKEN]:
            break

    # Convert indices to tokens (excluding <SOS> and <EOS>)
    decoded_tokens = [dgs_idx2token[str(idx)] for idx in tgt_indices if idx not in {dgs_token2idx[SOS_TOKEN], dgs_token2idx[EOS_TOKEN]}]
    return decoded_tokens

In [5]:
with open('dgs_vocab.json', 'r', encoding="utf-8") as f:
            vocab = json.load(f)
dgs_token2idx = vocab["token2idx"]
dgs_idx2token = vocab["idx2token"]
de_tokenizer = Tokenizer()

SRC_VOCAB_SIZE = de_tokenizer.vocab_size
TGT_VOCAB_SIZE = len(dgs_token2idx)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("German tokenizer vocab size:", de_tokenizer.vocab_size)
print("DGS vocab size:", len(dgs_token2idx))

German tokenizer vocab size: 31105
DGS vocab size: 4685


In [9]:
state_dir = "eval/candidates/check2"

# Model Configurations: 1       2       3       4       5       6       7       8       9
config = {"d":      [   64,     64,     64,     64  ],
          "el":     [   1,      1,      1,      1   ],
          "dl":     [   1,      1,      1,      1   ],
          "ff":     [   64,     64,     64,     64  ],
          "ep":     [   1000,   750,    500,    250 ]}

In [10]:
eval_file = "eval/inference/check2table.csv"
eval_file_printable = "eval/inference/check2printable.csv"

In [12]:
example_sentence = "Wie war dein Tag?"
use_testset = True

if use_testset:
    with open("example_sentences.txt", 'r') as f:
        sentences = [line.strip() for line in f]
    
    columns = ["Input"]
    for d, el, dl, ff, ep in zip(config["d"], config["el"], config["dl"], config["ff"], config['ep']):
        columns.append(f"d{d}_el{el}_dl{dl}_ff{ff}-ep{ep}")

    f = open(eval_file, "w", newline="", encoding='utf-8')
    writer_f = csv.writer(f, delimiter=";")
    writer_f.writerow(columns)

    fp = open(eval_file_printable, "w", newline="", encoding='utf-8')
    writer_fp = csv.writer(fp, delimiter=";")

else:
    sentences = [example_sentence]

for i, sentence in enumerate(sentences):
    print(f"  {i+1:3.0f}) Input sentence:\t\t{sentence}")
    if use_testset:
        writer_fp.writerow([f"#{i+1}", sentence])

    f_row = [sentence]

    for d, el, dl, ff, ep in zip(config["d"], config["el"], config["dl"], config["ff"], config['ep']):
        model_id = f"d{d}_el{el}_dl{dl}_ff{ff}-ep{ep}"
        model = Transformer(SRC_VOCAB_SIZE, TGT_VOCAB_SIZE, model_dim=d, num_heads=8,
                            num_encoder_layers=el, num_decoder_layers=dl,
                            ff_dim=ff, dropout=0).to(device)
        
        state_file = os.path.join(state_dir, f"{model_id}.pt")
            
        try:
            model.load_state_dict(torch.load(state_file, weights_only=True))
        except Exception as e:
            print("No saved state found. Skipping inference for version", model_id)

        decoded_dgs = decode(model, sentence, max_len=20,
                                    de_tokenizer=de_tokenizer,
                                    dgs_token2idx=dgs_token2idx, dgs_idx2token=dgs_idx2token, device=device)
        print(f"  {model_id}:\t{decoded_dgs}")

        if use_testset:
            dgs_string = str(decoded_dgs)
            f_row.append(dgs_string)
            writer_fp.writerow([model_id, dgs_string])

    if use_testset:
        writer_f.writerow(f_row)
    print("")

if use_testset:
    f.close()
    fp.close()


    1) Input sentence:		Guten Morgen!
  d64_el1_dl1_ff64-ep1000:	['als', 'ich', 'zeigen']
  d64_el1_dl1_ff64-ep750:	['zeigen', 'wie']
  d64_el1_dl1_ff64-ep500:	['zeigen', 'schütteln hand']
  d64_el1_dl1_ff64-ep250:	['prod', 'verschwommen']

    2) Input sentence:		Wie geht es dir?
  d64_el1_dl1_ff64-ep1000:	['wie frage vergleich']
  d64_el1_dl1_ff64-ep750:	['du']
  d64_el1_dl1_ff64-ep500:	['wie frage vergleich']
  d64_el1_dl1_ff64-ep250:	['wie frage vergleich', 'du']

    3) Input sentence:		Hast du gut geschlafen?
  d64_el1_dl1_ff64-ep1000:	['du', 'schlafen', 'du', 'schlafen', 'du']
  d64_el1_dl1_ff64-ep750:	['du', 'schlafen', 'du', 'schlafen', 'du']
  d64_el1_dl1_ff64-ep500:	['du', 'schlafen', 'du']
  d64_el1_dl1_ff64-ep250:	['du', 'schlafen', 'du']

    4) Input sentence:		Kannst du mir bitte helfen?
  d64_el1_dl1_ff64-ep1000:	['kann', 'bitte', 'helfen', 'thema']
  d64_el1_dl1_ff64-ep750:	['kann', 'helfen', 'thema']
  d64_el1_dl1_ff64-ep500:	['kann', 'helfen']
  d64_el1_dl1_ff64-ep2